In [ ]:
import vegas
import numpy as np

# --- THE FIX: Tell VEGAS to send 2D batches instead of 1D points ---
@vegas.batchintegrand
def target_pdf(x):
    # x is now guaranteed to be a batch of coordinates of shape (N_samples, 2)
    norm = 1.0 / (2.0 * np.pi * 0.2)
    
    # Peak 1 at (2, 2)
    diff1_sq = (x[:, 0] - 2.0)**2 + (x[:, 1] - 2.0)**2
    pdf1 = norm * np.exp(-diff1_sq / (2.0 * 0.2))
    
    # Peak 2 at (-2, -2)
    diff2_sq = (x[:, 0] + 2.0)**2 + (x[:, 1] + 2.0)**2
    pdf2 = norm * np.exp(-diff2_sq / (2.0 * 0.2))
    
    return 0.5 * pdf1 + 0.5 * pdf2

# The rest remains exactly the same
integ = vegas.Integrator([[-10.0, 10.0], [-10.0, 10.0]])

print("Starting VEGAS Adaptation with high statistics...")
integ(target_pdf, nitn=20, neval=100000)

print("Starting VEGAS Final Integration with massive statistics...")
# Runs 10 separate batches of 500,000 points, and averages them!
result = integ(target_pdf, nitn=10, neval=500000)

print("\n--- FINAL RESULTS ---")
print(result.summary())

integral = result.mean
error = result.sdev
variance = (error**2) * 50000

print(f"\nCalculated Integral : {integral:.6f} +/- {error:.6f}")
print(f"True Math Integral  : 1.000000")
print(f"Variance (sigma^2)  : {variance:.6f}")

In [ ]:
import vegas
import numpy as np

# Tell VEGAS to send us 2D batches for fast NumPy math
@vegas.batchintegrand
def target_pdf_vegas(x):
    # x[:, 0] is Energy (E) natively sampled from 50 to 150
    # x[:, 1] is cos(theta) natively sampled from -1 to 1
    E = x[:, 0]
    cos_theta = x[:, 1]
    
    m_Z = 91.1876
    Gamma_Z = 2.4952
    
    # The exact same Matrix Element math
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    breit_wigner = bw_numerator / bw_denominator
    
    angular = 1.0 + cos_theta**2
    
    # Return the raw physics probability (NOT log-space)
    return breit_wigner * angular

# ==========================================
# SET UP VEGAS WITH EXACT PHYSICAL LIMITS
# ==========================================
# [Energy Range], [Angle Range]
integ = vegas.Integrator([[50.0, 150.0], [-1.0, 1.0]])

# ==========================================
# TRAINING AND INTEGRATION
# ==========================================
print("Starting VEGAS Adaptation (Z Boson)...")
integ(target_pdf_vegas, nitn=30, neval=1000000)

print("\nStarting VEGAS Final Integration...")
# Using 10 iterations of 500,000 points to crush the error bar
result = integ(target_pdf_vegas, nitn=30, neval=1000000)

print("\n--- FINAL RESULTS ---")
print(result.summary())

integral = result.mean
error = result.sdev

# Total points = 10 iterations * 1,000,000 points per iteration
total_samples = 30 * 1000000 

# Calculate standard variance to compare with the PyTorch model
variance = (error**2) * total_samples

print(f"\nCalculated Integral : {integral:.6f} +/- {error:.6f}")
print(f"Variance (sigma^2)  : {variance:.6f}")

In [ ]:
import scipy.integrate as integrate

m_Z = 91.1876
Gamma_Z = 2.4952

# 1. Define only the Energy portion of the Breit-Wigner
def energy_integrand(E):
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    return bw_numerator / bw_denominator

# 2. Integrate with SciPy Quadrature between exactly 50 and 150
energy_integral, _ = integrate.quad(energy_integrand, 50.0, 150.0)

# 3. The exact analytical solution for the angular integral (-1 to 1)
angular_integral = 8.0 / 3.0  # (which is 2.6666...)

# 4. Multiply them back together
total_exact_integral = energy_integral * angular_integral

print(f"Energy Integral (50 to 150) : {energy_integral:.6f}")
print(f"Total Exact Integral        : {total_exact_integral:.6f}")

In [ ]:
import vegas
import numpy as np

# 1. Define the exact same 8D target function (in standard NumPy)
def rosenbrock_pdf(x):
    # x is a single 8-dimensional point: [x0, x1, x2, x3, x4, x5, x6, x7]
    x_np = np.asarray(x)
    
    # We slice the array just like we did in PyTorch
    x_i = x_np[:-1]    # The first 7 dimensions
    x_next = x_np[1:]  # The next 7 dimensions
    
    term1 = 100.0 * (x_next - x_i**2)**2
    term2 = (1.0 - x_i)**2
    
    # We use the exact same -0.05 scaling factor inside the exponent
    f_phys = np.exp(-0.05 * np.sum(term1 + term2))
    
    return f_phys

# 2. Initialize the VEGAS Integrator
# We tell it we have 8 dimensions, all bounded between -5.0 and +5.0
bounds = [[-5.0, 5.0]] * 8
integ = vegas.Integrator(bounds)

print("Step 1: Training the VEGAS grid (Warm-up)...")
# We give it 10 iterations of 100,000 points just to try and find the 8D ribbon
integ(rosenbrock_pdf, nitn=30, neval=1000000)

print("\nStep 2: Computing the final 8D integral...")
# We blast it with half a million points per iteration to get the final number
result = integ(rosenbrock_pdf, nitn=30, neval=1000000)

# 3. Print the brutal reality
print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
print(f"\nCalculated Integral : {result.mean:.6f} +/- {result.sdev:.6f}")
print(f"Relative Error      : {(result.sdev / result.mean) * 100:.2f} %")
print("=========================================")

In [ ]:
import vegas
import numpy as np

# 1. Define the 16D Topological Trap in pure NumPy
def topological_trap_pdf(x):
    # x is a single 16-dimensional point
    x_np = np.asarray(x)
    sigma = 0.2
    
    # Calculate the squared distance to Galaxy A (+2.5 on all axes)
    dist_A = np.sum((x_np - 2.5)**2)
    
    # Calculate the squared distance to Galaxy B (-2.5 on all axes)
    dist_B = np.sum((x_np + 2.5)**2)
    
    # VEGAS needs the raw probability (not the log probability)
    prob_A = np.exp(-0.5 * dist_A / sigma**2)
    prob_B = np.exp(-0.5 * dist_B / sigma**2)
    
    return prob_A + prob_B

# 2. Initialize VEGAS for 16 Dimensions
bounds = [[-5.0, 5.0]] * 16
integ = vegas.Integrator(bounds)

print("Step 1: VEGAS attempting to find the 16D galaxies (Warm-up)...")
# We give it 10 iterations of 100,000 darts to map the space
integ(topological_trap_pdf, nitn=10, neval=1000000)

print("\nStep 2: Computing the final 16D integral...")
# We blast it with half a million darts per iteration for the final test
result = integ(topological_trap_pdf, nitn=10, neval=5000000)

# 3. Print the brutal reality
print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
# Using scientific notation (.10e) because the numbers might be microscopically small
print(f"\nCalculated Integral : {result.mean:.10e} +/- {result.sdev:.10e}")
print("=========================================")

In [ ]:
import vegas
import numpy as np
import math

# ==========================================
# 1. THE QUANTUM DALITZ MANIFOLD
# ==========================================
def quantum_dalitz_pdf(x):
    # VEGAS hands us a point x in 4D space
    E1 = x[0]
    E2 = x[1]
    cos_theta = x[2]
    phi = x[3]
    
    # Kinematics (Invariant Masses)
    s12 = 2.0 * E1 * E2 * (1.0 - cos_theta)
    s23 = 2.0 * E1 * E2 * (1.0 + cos_theta)
    
    # Breit-Wigner Parameters
    M1_sq = 20.0
    G1_sq = 4.0
    M2_sq = 40.0
    G2_sq = 4.0
    
    bw1_denom = (s12 - M1_sq)**2 + G1_sq
    bw2_denom = (s23 - M2_sq)**2 + G2_sq
    
    mag_A1_sq = 1.0 / bw1_denom
    mag_A2_sq = 1.0 / bw2_denom
    
    # Quantum Interference Amplitudes
    real_A1 = (s12 - M1_sq) / bw1_denom
    imag_A1 = -math.sqrt(G1_sq) / bw1_denom
    
    real_A2 = ((s23 - M2_sq) * math.cos(phi) + math.sqrt(G2_sq) * math.sin(phi)) / bw2_denom
    imag_A2 = ((s23 - M2_sq) * math.sin(phi) - math.sqrt(G2_sq) * math.cos(phi)) / bw2_denom
    
    interference = 2.0 * (real_A1 * real_A2 + imag_A1 * imag_A2)
    
    # The total probability |M|^2
    matrix_element_sq = mag_A1_sq + mag_A2_sq + interference
    
    return matrix_element_sq

# ==========================================
# 2. VEGAS INTEGRATION ENGINE
# ==========================================
# E1 [0,10], E2 [0,10], cos(theta) [-1,1], phi [0, 2*pi]
bounds = [[0.0, 10.0], [0.0, 10.0], [-1.0, 1.0], [0.0, 2.0 * math.pi]]
integ = vegas.Integrator(bounds)

print("Step 1: VEGAS mapping the Quantum Manifold (Warm-up)...")
# VEGAS attempts to adapt its grid over 10 iterations
integ(quantum_dalitz_pdf, nitn=20, neval=300000)

print("\nStep 2: Computing the final 4D integral...")
# Blast it with half a million points per iteration
result = integ(quantum_dalitz_pdf, nitn=20, neval=1000000)

print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
print(f"\nCalculated Integral : {result.mean:.6f} +/- {result.sdev:.6f}")
print("=========================================")

In [2]:
import vegas
import math
import numpy as np

# ==========================================
# 1. THE 5-RESONANCE KINEMATIC MANIFOLD
# ==========================================
def grand_unified_dalitz_pdf(x):
    # VEGAS gives us 5 random variables between 0 and 1
    # We map them to the exact Autoregressive Kinematics we gave the AI
    
    # s12 ranges from 0.1 to 99.0
    s12 = 0.1 + 98.9 * x[0]
    
    # s23 width depends on s12 (Energy Conservation)
    s23_width = 99.9 - s12
    s23 = 0.1 + s23_width * x[1]
    
    # s13 is fully constrained
    s13 = 100.0 - s12 - s23
    
    # The 3 Quantum Angles
    cos_theta = 2.0 * x[2] - 1.0
    sin_theta_sq = 1.0 - cos_theta**2
    phi = 2.0 * math.pi * x[3]
    alpha = 2.0 * math.pi * x[4]
    
    # ---------------------------------------------------------
    # THE JACOBIAN (The phase-space stretching factor)
    # ---------------------------------------------------------
    # We must multiply by the volume of the space we stretched x into
    jacobian = 98.9 * s23_width * 2.0 * (2.0 * math.pi) * (2.0 * math.pi)
    
    # ---------------------------------------------------------
    # THE 5 COMPLEX BREIT-WIGNER AMPLITUDES
    # ---------------------------------------------------------
    # BW1: The Diagonal Slasher (s13)
    denom_1 = (s13 - 40.0)**2 + 16.0
    re_1 = (s13 - 40.0) / denom_1
    im_1 = -4.0 / denom_1
    
    # BW2 & BW3: The Quantum Braids (s12)
    denom_2 = (s12 - 30.0)**2 + 4.0
    re_2 = (s12 - 30.0) / denom_2
    im_2 = -2.0 / denom_2
    
    denom_3 = (s12 - 33.0)**2 + 2.25
    re_3 = ((s12 - 33.0) * math.cos(phi) + 1.5 * math.sin(phi)) / denom_3
    im_3 = ((s12 - 33.0) * math.sin(phi) - 1.5 * math.cos(phi)) / denom_3
    
    # BW4 & BW5: The Distant Exiles (s23)
    denom_4 = (s23 - 15.0)**2 + 1.0
    re_4 = sin_theta_sq * ((s23 - 15.0) / denom_4)
    im_4 = sin_theta_sq * (-1.0 / denom_4)
    
    denom_5 = (s23 - 80.0)**2 + 1.0
    re_5 = (math.cos(alpha)**2) * ((s23 - 80.0) / denom_5)
    im_5 = (math.cos(alpha)**2) * (-1.0 / denom_5)
    
    # ---------------------------------------------------------
    # QUANTUM SUPERPOSITION
    # ---------------------------------------------------------
    total_re = re_1 + re_2 + re_3 + re_4 + re_5
    total_im = im_1 + im_2 + im_3 + im_4 + im_5
    
    matrix_element_sq = total_re**2 + total_im**2
    
    # Return the un-logged physics probability * the Jacobian mapping
    return matrix_element_sq * jacobian

# ==========================================
# 2. VEGAS INTEGRATION ENGINE
# ==========================================
# We give VEGAS the base [0, 1] unit hypercube in 5 Dimensions
bounds = [[0.0, 1.0]] * 5
integ = vegas.Integrator(bounds)

print("Step 1: VEGAS adapting grid to the 5D Manifold (Warm-up)...")
# VEGAS attempts to adapt its grid over 10 iterations
integ(grand_unified_dalitz_pdf, nitn=20, neval=400000)

print("\nStep 2: Computing the final 5D integral...")
# Blast it with 2 million points per iteration
result = integ(grand_unified_dalitz_pdf, nitn=20, neval=3000000)

print("\n=========================================")
print("          FINAL VEGAS RESULTS            ")
print("=========================================")
print(result.summary())
print(f"\nCalculated Integral : {result.mean:.6f} +/- {result.sdev:.6f}")
print(f"Q-Value (Quality)   : {result.Q:.2f}")
print("=========================================")

Step 1: VEGAS adapting grid to the 5D Manifold (Warm-up)...

Step 2: Computing the final 5D integral...

          FINAL VEGAS RESULTS            
itn   integral        wgt average     chi2/dof        Q
-------------------------------------------------------
  1   38213(16)       38213(16)           0.00     1.00
  2   38224(11)       38220.1(9.3)        0.32     0.57
  3   38222(12)       38221.0(7.3)        0.17     0.84
  4   38218(12)       38220.1(6.2)        0.13     0.94
  5   38209(12)       38217.6(5.5)        0.28     0.89
  6   38220(12)       38218.1(4.9)        0.23     0.95
  7   38224(12)       38219.0(4.5)        0.23     0.97
  8   38206(12)       38217.3(4.2)        0.34     0.93
  9   38218(12)       38217.4(4.0)        0.30     0.97
 10   38218(12)       38217.4(3.8)        0.27     0.98
 11   38214(12)       38217.0(3.6)        0.25     0.99
 12   38215(12)       38216.8(3.4)        0.23     1.00
 13   38211(12)       38216.4(3.3)        0.23     1.00
 14   38213(1